Aquí van las librerías y dependencias

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


Aquí vamos a ubicar el chromedriver, es una herramienta para scrapping automatizado

In [2]:
# Ruta a tu chromedriver
CHROMEDRIVER_PATH = "D:/chromedriver/chromedriver.exe"  # <-- cambia si está en otro lugar

In [3]:
# Configurar navegador (puedes quitar headless si quieres ver el navegador)
chrome_options = Options()
chrome_options.add_argument("--start-maximized")


In [4]:
# Inicializar navegador
service = Service(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [5]:
# Ir al sitio principal
url = "https://www.epn.edu.ec/oferta-de-grado"
driver.get(url)
time.sleep(2)  # espera que cargue

In [21]:
# Extraer enlaces
el_tags = driver.find_elements(By.CLASS_NAME, "elementor elementor-18689")
carreras = []

palabras_excluir = ["tecnologia-superior", "intranet", "tecnólogo"]

for el in el_tags:
    a_tags = el.find_elements(By.TAG_NAME, "a")
    for a in a_tags:
        href = a.get_attribute("href")
        
        # Saltar si no hay href
        if not href:
            continue
        
        texto = a.text.strip()
        
        slug = href.strip("/").split("/")[-1].lower()
        
        if any(p in slug for p in palabras_excluir):
            continue
        
        carreras.append((texto, href))

In [22]:
# Mostrar resultados
for nombre, link in carreras:
    print(f"🎓 {nombre} → {link}")
print(f"\n🔢 Total de carreras de ingeniería encontradas: {len(carreras)}")


🔢 Total de carreras de ingeniería encontradas: 0


In [8]:
# ▶️ URL de una carrera (puedes probar con cualquiera de las que ya tienes)
url = "https://www.epn.edu.ec/oferta-academica/grado/ingenieria-tecnologia/carreras-de-grado/rra-administracion-de-empresas/"
driver.get(url)



In [9]:
# Esperar que aparezcan las pestañas
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "ul.nav-tabs.nav-justified"))
    )
    print("✅ Se encontró la sección 'Más información'")
except Exception as e:
    print("❌ No se encontró la sección 'Más información'")
    driver.quit()
    exit()

❌ No se encontró la sección 'Más información'


In [ ]:
# Diccionario para almacenar resultados
resultados = {
    "perfil_ingreso": "",
    "perfil_egreso": "",
    "perfil_profesional": "",
    "requisitos": ""
}

# Etiquetas visibles que aparecen en el menú de pestañas
pestañas = {
    "PERFIL DE INGRESO": "perfil_ingreso",
    "PERFIL DE EGRESO": "perfil_egreso",
    "PERFIL PROFESIONAL": "perfil_profesional",
    "REQUISITOS": "requisitos"
}

# Iterar por cada pestaña
for nombre_visible, clave in pestañas.items():
    try:
        tab = driver.find_element(By.XPATH, f"//ul[contains(@class, 'nav-tabs')]//a[contains(text(), '{nombre_visible}')]")
        tab.click()
        time.sleep(1)  # Esperar a que cambie el contenido

        # Extraer el contenido que aparece en el panel activo
        contenido = driver.find_element(By.CSS_SELECTOR, "div.tab-content .tab-pane.active")
        resultados[clave] = contenido.text.strip()
        print(f"✅ {nombre_visible} extraído")

    except Exception as e:
        print(f"⚠️ No se pudo extraer {nombre_visible}: {e}")
        resultados[clave] = ""

# Mostrar resultados
print("\n📋 Resultados:")
for k, v in resultados.items():
    print(f"\n🔹 {k.upper()}:\n{v[:500]}{'...' if len(v) > 500 else ''}")  # Mostrar un extracto

⚠️ No se pudo extraer PERFIL DE INGRESO: HTTPConnectionPool(host='localhost', port=64822): Max retries exceeded with url: /session/07444db5150812f924e2a9443403eb68/element (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002CF2D1B68E0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
⚠️ No se pudo extraer PERFIL DE EGRESO: HTTPConnectionPool(host='localhost', port=64822): Max retries exceeded with url: /session/07444db5150812f924e2a9443403eb68/element (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002CF2D1B68B0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
⚠️ No se pudo extraer PERFIL PROFESIONAL: HTTPConnectionPool(host='localhost', port=64822): Max retries exceeded with url: /session/07444db5150812f924e2a9443403eb68/element (Caused by NewConnection

: 

In [ ]:
# Finalizar
driver.quit()